In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud
import re


In [ ]:
df_fake = pd.read_csv("../data/raw/Fake.csv")
df_true = pd.read_csv("../data/raw/True.csv")

# Add labels
df_fake["label"] = 0   # 0 = fake
df_true["label"] = 1   # 1 = true

# Combine into one dataset
df = pd.concat([df_fake, df_true], ignore_index=True)

df.head()



In [ ]:
df.info()
df.columns
df.isnull().sum()



In [ ]:
plt.figure(figsize=(6,4))
sns.countplot(x=df["label"])
plt.title("Fake vs Real News Distribution")
plt.show()

df["label"].value_counts(normalize=True)


In [ ]:
df["text_length"] = df["text"].apply(lambda x: len(str(x).split()))
df["text_length"].describe()

plt.figure(figsize=(7,4))
sns.histplot(df["text_length"], bins=50)
plt.title("Text Length Distribution")
plt.show()


In [ ]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r"[^a-zA-Z\s]", "", text)
    return text


In [ ]:
from collections import Counter

all_words = " ".join(df["text"].apply(clean_text)).split()
word_freq = Counter(all_words)

word_freq.most_common(20)

common_words = dict(word_freq.most_common(20))

plt.figure(figsize=(10,5))
sns.barplot(x=list(common_words.values()), y=list(common_words.keys()))
plt.title("Top 20 Words")
plt.show()


In [ ]:
text_string = " ".join(df["text"].astype(str))

wordcloud = WordCloud(width=800, height=400).generate(text_string)

plt.figure(figsize=(10,5))
plt.imshow(wordcloud, interpolation="bilinear")
plt.axis("off")
plt.show()


In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer(ngram_range=(2,2), max_features=20)
X = vectorizer.fit_transform(df["text"].astype(str))

bigrams = vectorizer.get_feature_names_out()

counts = X.toarray().sum(axis=0)

plt.figure(figsize=(10,5))
sns.barplot(x=counts, y=bigrams)
plt.title("Top 20 Bigrams")
plt.show()



In [ ]:
df_clean = df.copy()
df_clean["text"] = df_clean["text"].apply(clean_text)

df_clean.to_csv("../data/processed/news_clean.csv", index=False)
